In [ ]:
# start coding here
import logging
import json
import os
from collections import defaultdict

import seaborn as sns
import pandas as pd
import anndata

import numpy as np


# setup snakemake logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(snakemake.log[0]), logging.StreamHandler()],  # type: ignore [reportUndefinedVariable]
)


In [ ]:
reviews = []
with open(snakemake.input.evaluation) as f:
    for review_str in f:
        review = json.loads(review_str)
        review["llava_score"] = review["tuple"][0]
        review["reference_score"] = review["tuple"][1]
        review["dataset"] = "archs4_metasra" if "SRX" in review["question_id"] else "cellxgene_census"
        del review["tuple"]
        reviews.append(review)
df = pd.DataFrame(reviews)        

In [ ]:
df["normed"] = df["llava_score"] / df["reference_score"]


In [ ]:
# plot it

In [ ]:
sns.violinplot(data=df, x="normed", y="dataset")

In [ ]:
df[df.normed < 0.5].iloc[5]["content"]


In [ ]:
import anndata
adata = anndata.read_h5ad(snakemake.input.archs4_data, backed="r")


In [ ]:
single_cells = adata.obs.query("singlecellprobability > 0.1").index  # TODO can also use 0.1 (not much difference)

In [ ]:
single_cells

In [ ]:
df["sample_id"] = df.question_id.apply(lambda v: v.split("_", maxsplit=1)[1])

In [ ]:
df["singlecell"] = df["sample_id"].isin(single_cells)

In [ ]:
df["is_complex"] = df["sample_id"].isin(snakemake.params.complex_samples)
df["is_detailed"] = df["sample_id"].isin(snakemake.params.detailed_samples)
df["group"] = df.apply(lambda row: "detailed question" if row.is_detailed else ("complex question" if row.is_complex else "normal question"), axis=1)

In [ ]:
df["is_detailed"].value_counts()

In [ ]:
df[df.normed < 0.4].loc[2]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 2))
sns.barplot(data=df, x="normed", y="dataset", hue="group", ax=ax)
ax.set(title=f"Mean score: {df.normed.mean():.2f}")
sns.despine()

plt.tight_layout()
fig.savefig(snakemake.output.overview_plot)
fig.savefig(snakemake.output.overview_plot + ".png")

logging.info(f"Overall score: {df.normed.mean()}. without single cells: {df[~df.singlecell].normed.mean()}")

# the presence of single cells in the training dataset overall impacted these scores only minorly (exclusion of 'cells with single cell probability > 0.1' in test set: 0.65 -> 0.63)
# failure modes: real hallucination. 'honest' mistakes


In [ ]:
sns.barplot(data=df[~df.singlecell], y="normed", x="dataset", hue="group")